# Notebook 3: Fine-Tune Best Settings (Round 2 Optimization)

**For beginners.** This notebook will:
1. Read the best settings from Round 1
2. Try small changes around those best settings
3. Find the perfect settings in about 4 hours

**Why Round 2?** Round 1 searched everywhere. Round 2 looks closely around the best spot.

**Estimated time:** 4 hours on A100 GPU

**Prerequisite:** Finish `02_optimize_round1.ipynb` first.


## Step 1: Check GPU and Set Memory Limit

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.utils.env import set_gpu_memory_fraction

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    set_gpu_memory_fraction(0.8)
else:
    print("❌ No GPU!")

## Step 2: Set HuggingFace Token

In [ ]:
import os

os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
print("✅ Token set")

## Step 3: Load Round 1 Results

This reads the best settings found in notebook 02.

In [ ]:
import json

round1_path = '../artifacts/optuna/round1_best_params.json'

if not os.path.exists(round1_path):
    print("❌ ERROR: Round 1 results not found!")
    print("   Please run notebook 02 first.")
else:
    with open(round1_path, 'r') as f:
        round1_data = json.load(f)
    
    best_params = round1_data['best_params']
    print("✅ Round 1 results loaded!")
    print(f"   Best loss: {round1_data['best_value']:.4f}")
    print("   Best settings:")
    for key, value in best_params.items():
        print(f"     {key}: {value}")

## Step 4: Create Narrow Search Space

Instead of searching everywhere, we only try values CLOSE to the Round 1 winner.

For example, if Round 1 found learning_rate = 0.0002, Round 2 will try 0.00014 to 0.00026.


In [ ]:
import numpy as np

def create_narrow_space(best_params, variance=0.3):
    space = {}
    for key, value in best_params.items():
        if key == 'learning_rate':
            log_val = np.log10(value)
            space[key] = [10**(log_val - variance), 10**(log_val + variance)]
        elif key in ['weight_decay', 'lora_dropout', 'warmup_ratio']:
            space[key] = [max(0, value * 0.7), min(1, value * 1.3)]
        elif key == 'temperature':
            space[key] = [max(1.0, value * 0.7), value * 1.3]
        elif key in ['alpha', 'beta']:
            space[key] = [max(0.1, value * 0.7), min(0.9, value * 1.3)]
        elif key in ['lora_r', 'lora_alpha']:
            space[key] = [max(4, value - 8), value + 8]
        elif key in ['per_device_train_batch_size', 'gradient_accumulation_steps', 'num_train_epochs']:
            space[key] = [value]  # Keep the same
        elif key == 'max_length':
            space[key] = [256, 512, 1024]
    return space

narrow_space = create_narrow_space(best_params)
print("🔍 Narrow search space created:")
for key, bounds in narrow_space.items():
    print(f"   {key}: {bounds}")

## Step 5: Load Data and Teacher Model

In [ ]:
from src.config import load_config
from src.data.dataset_loader import load_and_prepare_dataset
from src.data.preprocessing import preprocess_dataset
from src.data.tokenization import tokenize_dataset, get_tokenizer
from src.models.teacher_loader import load_teacher_model

config = load_config('../configs/default.yaml')

print("📥 Loading dataset...")
dataset = load_and_prepare_dataset(config.dataset, seed=42)
dataset = preprocess_dataset(dataset, config.dataset)

print("🧠 Loading teacher model...")
teacher, _ = load_teacher_model(config.models['teacher'])

student_tokenizer = get_tokenizer(config.models['student'].name)
tokenized_dataset = tokenize_dataset(dataset, student_tokenizer, config.tokenization)

print("✅ Ready!")

## Step 6: Run Round 2 Optimization

This tries 30 experiments near the best Round 1 settings.

**What you will see:**
```
--- Trial 0 ---
Trying: lr=0.00018, lora_r=14, temp=2.1
Trial 0 done. Loss: 1.95
```

If the loss is lower than Round 1, we found something better! 🎉


In [ ]:
from src.models.student_loader import load_student_model
from src.data.collators import DataCollatorForCausalLM
from src.models.distillation import create_distillation_trainer
from src.optimization.optuna_search import run_optuna_optimization, save_best_params
from src.config import Config
import optuna
import gc

n_trials = 30
print(f"🔬 Starting Round 2 with {n_trials} trials...")
print("This will take about 4 hours.")
print("=" * 50)

def objective(trial: optuna.Trial) -> float:
    # Pick values from narrow space
    params = {}
    for key, bounds in narrow_space.items():
        if len(bounds) == 1:
            params[key] = bounds[0]
        elif key == 'learning_rate':
            params[key] = trial.suggest_float(key, bounds[0], bounds[1], log=True)
        elif key in ['lora_r', 'lora_alpha']:
            params[key] = trial.suggest_int(key, bounds[0], bounds[1])
        elif key in ['per_device_train_batch_size', 'gradient_accumulation_steps', 'num_train_epochs', 'max_length']:
            params[key] = trial.suggest_categorical(key, bounds)
        else:
            params[key] = trial.suggest_float(key, bounds[0], bounds[1])
    
    # Make sure alpha + beta = 1.0
    if 'alpha' in params and 'beta' in params:
        total = params['alpha'] + params['beta']
        params['alpha'] /= total
        params['beta'] /= total
    
    print(f"\n--- Trial {trial.number} ---")
    print("Trying settings close to Round 1 winner...")
    
    trial_config = Config.from_dict(config.to_dict())
    trial_config.training.learning_rate = params['learning_rate']
    trial_config.training.weight_decay = params.get('weight_decay', best_params.get('weight_decay', 0.01))
    trial_config.training.num_train_epochs = min(params.get('num_train_epochs', 2), 2)
    trial_config.training.warmup_ratio = params.get('warmup_ratio', best_params.get('warmup_ratio', 0.03))
    trial_config.training.per_device_train_batch_size = params.get('per_device_train_batch_size', 1)
    trial_config.training.gradient_accumulation_steps = params.get('gradient_accumulation_steps', 8)
    trial_config.lora.r = params.get('lora_r', 16)
    trial_config.lora.lora_alpha = params.get('lora_alpha', 32)
    trial_config.lora.lora_dropout = params.get('lora_dropout', 0.05)
    trial_config.distillation.temperature = params.get('temperature', 2.0)
    trial_config.distillation.alpha = params.get('alpha', 0.3)
    trial_config.distillation.beta = params.get('beta', 0.7)
    trial_config.tokenization.max_length = params.get('max_length', 512)
    
    try:
        student, _ = load_student_model(
            trial_config.models['student'],
            lora_config=trial_config.lora
        )
        
        data_collator = DataCollatorForCausalLM(
            tokenizer=student_tokenizer,
            max_length=trial_config.tokenization.max_length
        )
        
        trial_dir = f"../artifacts/optuna/round2_trial_{trial.number}"
        trainer = create_distillation_trainer(
            student_model=student,
            teacher_model=teacher,
            train_dataset=tokenized_dataset['train'],
            eval_dataset=tokenized_dataset['validation'],
            tokenizer=student_tokenizer,
            data_collator=data_collator,
            output_dir=trial_dir,
            temperature=trial_config.distillation.temperature,
            alpha=trial_config.distillation.alpha,
            beta=trial_config.distillation.beta,
            num_train_epochs=trial_config.training.num_train_epochs,
            per_device_train_batch_size=trial_config.training.per_device_train_batch_size,
            gradient_accumulation_steps=trial_config.training.gradient_accumulation_steps,
            learning_rate=trial_config.training.learning_rate,
            weight_decay=trial_config.training.weight_decay,
            warmup_ratio=trial_config.training.warmup_ratio,
            logging_steps=50,
            eval_steps=100,
            save_steps=1000,
            bf16=trial_config.hardware.mixed_precision == 'bf16',
            fp16=trial_config.hardware.mixed_precision == 'fp16',
            dataloader_num_workers=trial_config.training.dataloader_num_workers,
            gradient_checkpointing=trial_config.hardware.gradient_checkpointing,
        )
        
        trainer.train()
        metrics = trainer.evaluate()
        eval_loss = metrics.get('eval_loss', float('inf'))
        
        print(f"Trial {trial.number} done. Loss: {eval_loss:.4f}")
        
        del student
        del trainer
        gc.collect()
        torch.cuda.empty_cache()
        
        return eval_loss
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return float('inf')

study = run_optuna_optimization(
    objective_func=objective,
    study_name='kd_round2_notebook',
    n_trials=n_trials,
    timeout=18000,  # 5 hours max
    direction='minimize',
    seed=42,
    show_progress_bar=True
)

print("\n" + "=" * 50)
print("🎉 ROUND 2 COMPLETE!")
print(f"Best loss: {study.best_value:.4f}")
print(f"Best settings: {study.best_params}")
print(f"\nRound 1 best: {round1_data['best_value']:.4f}")
print(f"Round 2 best: {study.best_value:.4f}")
improvement = round1_data['best_value'] - study.best_value
print(f"Improvement: {improvement:.4f}")

## Step 7: Save Final Best Settings

In [ ]:
save_best_params(
    study,
    '../artifacts/optuna/round2_best_params.json'
)

print("✅ Final best settings saved!")
print("\nYou can now train the final model using these best settings.")
print("Run notebook 01 again with the best config, or use the CLI.")